# C2 classifier — fine-tune for permission classification on Colab

Launcher for the `mostargate.classifier.train` module. The training code lives in `mostargate/classifier/train.py` in the repo; this notebook just bootstraps a Colab T4 instance and runs that module.

**Workflow:**
1. `Runtime → Change runtime type → T4 GPU` (free tier).
2. `Runtime → Run all`.
3. When training finishes, download `classifier_model_roberta.zip` from the **Files** panel on the left, unzip locally into `dataset/classifier_artifacts/`.

Default config: `roberta-base`, 40 epochs, LR 2e-5, batch 16, fp32. Wall time on T4: ~5–7 minutes total. See `docs/phase_c_classifier_findings.md` §7 for results.

An optional `roberta-large` cell is included below (commented out) — try that if you want to test the higher-capacity variant. Expect ~3× training time and a modest precision/F1 improvement.

## 1. Pull the repo

Always start by `%cd /content` — this resets the working directory so re-runs after a previous clone don't create nested `mostargate/mostargate/` matryoshka folders or fail to remove the directory you're inside.

In [ ]:
%cd /content
!rm -rf mostargate
!git clone https://github.com/0xballistics/mostargate.git
%cd mostargate

## 2. Install Python deps

Colab images already include `torch` and `numpy`. We add the few packages `mostargate.classifier.train` imports beyond that.

In [ ]:
!pip install -q transformers datasets sentencepiece protobuf python-dotenv

## 3. Verify GPU is attached

In [ ]:
!nvidia-smi

## 4. Train — roberta-base (default)

Canonical Phase C settings per `docs/phase_c_classifier_findings.md` §7.3:
- 40 epochs, batch 16, LR 2e-5, max_len 256, seed 42
- BCEWithLogitsLoss with per-class `pos_weight` clipped at 10.0
- `max_grad_norm` 1.0
- fp32 (fp16 disabled by default due to a transformers 5.x grad-unscale regression)
- No internal validation hold-out — all 500 train records used for gradient updates.

First run downloads roberta-base (~500 MB) from the HuggingFace hub. Subsequent re-runs in the same session use the cached download.

In [ ]:
!python -m mostargate.classifier.train --epochs 40 --learning-rate 2e-5

## 4b. (Optional) Train — roberta-large variant

`roberta-large` is ~3× larger (355M params) than `roberta-base`. It uses the **same architecture and hyperparameters**, so it tolerates LR 2e-5 and the same `pos_weight` settings.

**Expected impact** (versus roberta-base at 40 epochs):
- Training time: ~15 minutes on T4 (vs 5 min for base)
- Macro-F1: likely +2–4 pp (closing some of the gap to Claude's 0.886)
- Undershoot: likely -3–5 pp (the main bottleneck per §7.8 of the findings doc)
- Macro-P: probably similar or slightly higher

**Tradeoff**: 3× larger model means ~3× slower CPU inference at deployment time and ~3× more disk for the saved weights. If results materially exceed roberta-base, document and switch; if marginal, roberta-base remains the canonical model.

Skip cell 4 above (roberta-base training) and uncomment + run this cell instead. The trained model overwrites `dataset/classifier_artifacts/model/`.

In [ ]:
# Uncomment to train roberta-large instead of roberta-base:
# !python -m mostargate.classifier.train --model roberta-large --epochs 40 --learning-rate 2e-5

## 5. Test the trained model — per-permission F1 at threshold 0.5

Loads the saved model, runs inference on the 100-record test set, prints probability distribution and per-permission F1. The `probs` variable produced here is reused by the threshold-sweep cell below.

In [ ]:
import json, torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from mostargate.constants import TOOLS

PERMISSIONS = list(TOOLS.keys())
MODEL_DIR  = "dataset/classifier_artifacts/model"

tok   = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).cuda().eval()

test  = json.load(open("dataset/test.json"))
enc   = tok([r["prompt"] for r in test], padding=True, truncation=True,
            max_length=256, return_tensors="pt").to("cuda")
with torch.no_grad():
    probs = torch.sigmoid(model(**enc).logits).cpu().numpy()

print(f"shape={probs.shape}  mean={probs.mean():.3f}  std={probs.std():.3f}\n")
for lo, hi in [(0,.05),(.05,.20),(.20,.40),(.40,.60),(.60,.80),(.80,.95),(.95,1.01)]:
    n   = ((probs >= lo) & (probs < hi)).sum()
    pct = 100*n/probs.size
    print(f"[{lo:.2f}, {hi:.2f}): {pct:5.1f}%  {'#'*int(pct/2)}")

gt = np.array([[r["permissions"].get(p, False) for p in PERMISSIONS] for r in test], dtype=int)
preds = (probs >= 0.5).astype(int)
print(f"\n{'permission':<22}{'gt+':>5}{'pred+':>7}{'TP':>4}{'FP':>4}{'FN':>4}{'P':>6}{'R':>6}{'F1':>6}")
for j, p in enumerate(PERMISSIONS):
    tp = ((gt[:,j]==1) & (preds[:,j]==1)).sum()
    fp = ((gt[:,j]==0) & (preds[:,j]==1)).sum()
    fn = ((gt[:,j]==1) & (preds[:,j]==0)).sum()
    P  = tp / max(tp+fp, 1); R = tp / max(tp+fn, 1)
    F1 = 2*P*R / max(P+R, 1e-9)
    print(f"{p:<22}{gt[:,j].sum():>5}{preds[:,j].sum():>7}{tp:>4}{fp:>4}{fn:>4}{P:>6.2f}{R:>6.2f}{F1:>6.2f}")

macro_f1 = np.mean([
    (lambda tp,fp,fn: 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0)(
        ((gt[:,j]==1) & (preds[:,j]==1)).sum(),
        ((gt[:,j]==0) & (preds[:,j]==1)).sum(),
        ((gt[:,j]==1) & (preds[:,j]==0)).sum(),
    )
    for j in range(15)
])
print(f"\nmacro-F1 at threshold 0.5: {macro_f1:.3f}")
print(f"(reference: TF-IDF=0.70, Claude Haiku=0.89)")

## 6. Threshold sweep — six C2 configurations

Applies the six threshold configurations from `mostargate.classifier.baselines.THRESHOLD_CONFIGS` to the cached prediction matrix. Saves `roberta_test_probs.npy` so the matrix can be re-thresholded later without re-running inference.

In [ ]:
import json, numpy as np
from pathlib import Path
from mostargate.constants import TOOLS, TOOL_TIERS

PERMISSIONS = list(TOOLS.keys())

Path("dataset/classifier_artifacts").mkdir(parents=True, exist_ok=True)
np.save("dataset/classifier_artifacts/roberta_test_probs.npy", probs)
print(f"Saved probability matrix → roberta_test_probs.npy ({probs.shape})")

def tier_thr(p, t1, t2, t3):
    return {1: t1, 2: t2, 3: t3}[TOOL_TIERS[p]]

CONFIGS = {
    "static_05":             {p: 0.5 for p in PERMISSIONS},
    "static_08":             {p: 0.8 for p in PERMISSIONS},
    "risk_based_07_05_03":   {p: tier_thr(p, 0.7, 0.5, 0.3) for p in PERMISSIONS},
    "risk_based_06_04_02":   {p: tier_thr(p, 0.6, 0.4, 0.2) for p in PERMISSIONS},
    "risk_based_05_03_01":   {p: tier_thr(p, 0.5, 0.3, 0.1) for p in PERMISSIONS},
    "risk_based_08_06_04":   {p: tier_thr(p, 0.8, 0.6, 0.4) for p in PERMISSIONS},
}

W = {1: 3, 2: 2, 3: 1}
weights_per_perm = np.array([W[TOOL_TIERS[p]] for p in PERMISSIONS])
test = json.load(open("dataset/test.json"))
gt   = np.array([[r["permissions"].get(p, False) for p in PERMISSIONS] for r in test], dtype=int)

def metrics(preds):
    over_mask  = (preds == 1) & (gt == 0)
    under_mask = (preds == 0) & (gt == 1)
    sev_d  = (over_mask * weights_per_perm).sum(axis=1).mean()
    o_rate = over_mask.any(axis=1).mean()
    u_rate = under_mask.any(axis=1).mean()
    auto   = ~under_mask.any(axis=1)
    sev_auto = (over_mask[auto] * weights_per_perm).sum(axis=1).mean() if auto.any() else 0
    tp = ((gt == 1) & (preds == 1)).sum(axis=0)
    fp = ((gt == 0) & (preds == 1)).sum(axis=0)
    fn = ((gt == 1) & (preds == 0)).sum(axis=0)
    P = np.array([tp[j] / max(tp[j]+fp[j], 1) for j in range(15)])
    R = np.array([tp[j] / max(tp[j]+fn[j], 1) for j in range(15)])
    F1 = np.array([2*P[j]*R[j]/max(P[j]+R[j], 1e-9) for j in range(15)])
    return sev_d, o_rate, u_rate, P.mean(), F1.mean(), int(auto.sum()), sev_auto

print(f"\n{'config':<24}{'sev-d':>7}{'over':>7}{'under':>7}{'macroP':>9}{'macroF1':>9}{'auto':>6}{'sev/auto':>10}")
print('-' * 79)
print(f"{'C0 (all-grant ref)':<24}{'26.12':>7}{'100%':>7}{'0%':>7}{'—':>9}{'—':>9}{100:>6}{'26.12':>10}")
print(f"{'C1 (role ceiling)':<24}{'17.51':>7}{'100%':>7}{'2%':>7}{'—':>9}{'—':>9}{98:>6}{'~17.5':>10}")

for name, thr_map in CONFIGS.items():
    thr_vec = np.array([thr_map[p] for p in PERMISSIONS])
    preds = (probs >= thr_vec).astype(int)
    sev_d, o, u, mP, mF1, auto, sev_a = metrics(preds)
    star = " ★" if (u < 0.10 and mP >= 0.89) else ""
    print(f"{name:<24}{sev_d:>7.2f}{o:>7.1%}{u:>7.1%}{mP:>9.3f}{mF1:>9.3f}{auto:>6}{sev_a:>10.2f}{star}")

## 7. Bottleneck check — sweep excluding `email_send_external`

Per `docs/phase_c_classifier_findings.md` §7.8, `email_send_external` is the classifier-hardest permission (F1 ≈ 0.59) and also the trifecta-completing permission. Re-running the sweep with it excluded shows the metrics for the 14-permission system that a deterministic-rule-for-email_send_external hybrid architecture would deliver.

In [ ]:
import json, numpy as np
from mostargate.constants import TOOLS, TOOL_TIERS

PERMISSIONS = list(TOOLS.keys())
EXCLUDE = "email_send_external"
keep_idx = [j for j, p in enumerate(PERMISSIONS) if p != EXCLUDE]
keep_perms = [PERMISSIONS[j] for j in keep_idx]

probs_keep = probs[:, keep_idx]
test = json.load(open("dataset/test.json"))
gt_full = np.array([[r["permissions"].get(p, False) for p in PERMISSIONS] for r in test], dtype=int)
gt_keep = gt_full[:, keep_idx]
weights = np.array([{1:3,2:2,3:1}[TOOL_TIERS[p]] for p in keep_perms])

def tier_thr(p, t1, t2, t3):
    return {1: t1, 2: t2, 3: t3}[TOOL_TIERS[p]]
CONFIGS = {
    "static_05":             {p: 0.5 for p in keep_perms},
    "static_08":             {p: 0.8 for p in keep_perms},
    "risk_based_07_05_03":   {p: tier_thr(p, 0.7, 0.5, 0.3) for p in keep_perms},
    "risk_based_06_04_02":   {p: tier_thr(p, 0.6, 0.4, 0.2) for p in keep_perms},
    "risk_based_05_03_01":   {p: tier_thr(p, 0.5, 0.3, 0.1) for p in keep_perms},
    "risk_based_08_06_04":   {p: tier_thr(p, 0.8, 0.6, 0.4) for p in keep_perms},
}

def metrics(preds):
    over_mask  = (preds == 1) & (gt_keep == 0)
    under_mask = (preds == 0) & (gt_keep == 1)
    sev_d  = (over_mask * weights).sum(axis=1).mean()
    o_rate = over_mask.any(axis=1).mean()
    u_rate = under_mask.any(axis=1).mean()
    auto   = ~under_mask.any(axis=1)
    sev_auto = (over_mask[auto] * weights).sum(axis=1).mean() if auto.any() else 0
    tp = ((gt_keep == 1) & (preds == 1)).sum(axis=0)
    fp = ((gt_keep == 0) & (preds == 1)).sum(axis=0)
    fn = ((gt_keep == 1) & (preds == 0)).sum(axis=0)
    N = len(keep_perms)
    P = np.array([tp[j] / max(tp[j]+fp[j], 1) for j in range(N)])
    R = np.array([tp[j] / max(tp[j]+fn[j], 1) for j in range(N)])
    F1 = np.array([2*P[j]*R[j]/max(P[j]+R[j], 1e-9) for j in range(N)])
    return sev_d, o_rate, u_rate, P.mean(), F1.mean(), int(auto.sum()), sev_auto

print(f"Excluding {EXCLUDE!r}: {len(keep_perms)} permissions remain")
print(f"\n{'config':<24}{'sev-d':>7}{'over':>7}{'under':>7}{'macroP':>9}{'macroF1':>9}{'auto':>6}{'sev/auto':>10}")
print('-' * 79)
for name, thr_map in CONFIGS.items():
    thr_vec = np.array([thr_map[p] for p in keep_perms])
    preds = (probs_keep >= thr_vec).astype(int)
    sev_d, o, u, mP, mF1, auto, sev_a = metrics(preds)
    star = " ★" if (u < 0.10 and mP >= 0.89) else ""
    print(f"{name:<24}{sev_d:>7.2f}{o:>7.1%}{u:>7.1%}{mP:>9.3f}{mF1:>9.3f}{auto:>6}{sev_a:>10.2f}{star}")

## 8. Zip the trained model + cached probs for download

Bundles both the trained model directory and the cached prediction matrix into a single zip. Download from the Files panel on the left, then unzip locally so files land under `dataset/classifier_artifacts/`.

In [ ]:
!zip -qr classifier_model_roberta.zip \
    dataset/classifier_artifacts/model \
    dataset/classifier_artifacts/roberta_test_probs.npy
!ls -lh classifier_model_roberta.zip
print("\nDownload classifier_model_roberta.zip from the file panel (left sidebar).")